# Wafer Map Defect Classification — Part 1: Data, EDA and Baseline CNN
**Paper:** *An Explainable Hybrid CNN-Transformer Framework for Semiconductor Wafer Map Defect Classification*

**Before you run (Google Colab):** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

This notebook covers Week 1 and Week 2 of the plan:
1. Download WM-811K (no Kaggle login needed)
2. Clean the labels and explore the data (figures for your paper)
3. Preprocess: resize to 64×64, balance classes, split 70/15/15
4. Train a baseline CNN and save every result (accuracy, F1, confusion matrix, model size, inference time)

Every figure and table is saved in the `results/` folder. Download it at the end.

In [ ]:
!pip -q install kagglehub opencv-python-headless
import os, time, json
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
os.makedirs('results', exist_ok=True)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Download and load WM-811K
About 150 MB download. Loading the 2.1 GB pickle takes about 1 minute.

In [ ]:
import kagglehub
path = kagglehub.dataset_download('qingyi/wm811k-wafer-map')
df = pd.read_pickle(os.path.join(path, 'LSWMD.pkl'))
print(df.shape)
df.head()

## 2. Clean labels
Labels are stored as nested arrays like `[['Center']]`. Only about 21% of wafers are labelled; we keep those.

In [ ]:
def unwrap(x):
    try:
        return x[0][0] if len(x) > 0 and len(x[0]) > 0 else None
    except Exception:
        return None

df['failureType'] = df['failureType'].apply(unwrap)
df['trianTestLabel'] = df['trianTestLabel'].apply(unwrap)
lab = df[df['failureType'].notna()].reset_index(drop=True)
del df

CLASSES = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']
print('Labelled wafers:', len(lab))
counts = lab['failureType'].value_counts().reindex(CLASSES)
counts

## 3. Exploratory analysis (Figures 1–3 for the paper)

In [ ]:
# Fig: class distribution (log scale shows the heavy imbalance)
plt.figure(figsize=(9, 4))
sns.barplot(x=counts.index, y=counts.values, color='steelblue')
plt.yscale('log'); plt.ylabel('Number of wafers (log scale)'); plt.xlabel('Defect class')
plt.title('WM-811K class distribution (labelled wafers)')
for i, v in enumerate(counts.values): plt.text(i, v, str(v), ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.savefig('results/fig_class_distribution.png', dpi=300); plt.show()

# Fig: one example per class
fig, ax = plt.subplots(1, 9, figsize=(18, 2.4))
for i, c in enumerate(CLASSES):
    m = lab[lab['failureType'] == c]['waferMap'].iloc[0]
    ax[i].imshow(m, cmap='viridis'); ax[i].set_title(c); ax[i].axis('off')
plt.tight_layout(); plt.savefig('results/fig_samples.png', dpi=300); plt.show()

# Fig: original wafer map sizes vary, which is why we resize
sizes = lab['waferMap'].apply(lambda m: m.shape[0] * m.shape[1])
plt.figure(figsize=(6, 3.5)); plt.hist(sizes, bins=60, color='gray')
plt.xlabel('Pixels per wafer map'); plt.ylabel('Count'); plt.title('Wafer map size distribution')
plt.tight_layout(); plt.savefig('results/fig_size_distribution.png', dpi=300); plt.show()

## 4. Preprocessing
- Resize every map to **64×64** with nearest-neighbour (keeps values 0 = background, 1 = good die, 2 = defective die)
- One-hot encode the 3 values into 3 channels
- **Undersample `none`** to 10,000 wafers (it is about 85% of the data and would dominate training)
- Stratified split: 70% train, 15% validation, 15% test

In [ ]:
IMG = 64
NONE_KEEP = 10000

none_idx = lab.index[lab['failureType'] == 'none']
keep_none = np.random.choice(none_idx, NONE_KEEP, replace=False)
use = lab.loc[(lab['failureType'] != 'none') | lab.index.isin(keep_none)].reset_index(drop=True)

X = np.stack([cv2.resize(m.astype(np.uint8), (IMG, IMG), interpolation=cv2.INTER_NEAREST) for m in use['waferMap']])
X = np.eye(3, dtype=np.uint8)[np.clip(X, 0, 2)]            # (N, 64, 64, 3) one-hot, uint8 saves RAM
y = use['failureType'].map({c: i for i, c in enumerate(CLASSES)}).values
del use, lab; import gc; gc.collect()

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print('train', X_tr.shape, '| val', X_va.shape, '| test', X_te.shape)
pd.DataFrame({'train': np.bincount(y_tr, minlength=9), 'val': np.bincount(y_va, minlength=9),
              'test': np.bincount(y_te, minlength=9)}, index=CLASSES)

## 5. Augment rare classes (training set only)
Rare classes (Near-full, Donut, Random, Scratch) are grown to at least 2,000 samples using 90°/180°/270° rotations and flips. These are physically valid, because a defect pattern doesn't change meaning when the wafer is rotated.

In [ ]:
TARGET = 2000
def augment(img, k):
    img = np.rot90(img, k % 4)
    return np.fliplr(img) if k >= 4 else img

Xa, ya = [X_tr], [y_tr]
for c in range(len(CLASSES)):
    idx = np.where(y_tr == c)[0]
    need = TARGET - len(idx)
    if need <= 0: continue
    pick = np.random.choice(idx, need, replace=True)
    ks = np.random.randint(1, 8, need)
    Xa.append(np.stack([augment(X_tr[i], k) for i, k in zip(pick, ks)]))
    ya.append(np.full(need, c))
X_tr_aug = np.concatenate(Xa); y_tr_aug = np.concatenate(ya)
perm = np.random.permutation(len(y_tr_aug)); X_tr_aug, y_tr_aug = X_tr_aug[perm], y_tr_aug[perm]
print('Augmented train:', X_tr_aug.shape)
pd.Series(np.bincount(y_tr_aug, minlength=9), index=CLASSES)

## 6. Baseline CNN
A simple 4-block CNN. This is the baseline your hybrid model must beat in Part 2.

In [ ]:
def build_cnn(n_classes=9):
    L = tf.keras.layers
    inp = L.Input((IMG, IMG, 3))
    x = inp
    for f in [32, 64, 128, 256]:
        x = L.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = L.BatchNormalization()(x)
        x = L.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = L.MaxPooling2D()(x)
        x = L.Dropout(0.25)(x)
    x = L.GlobalAveragePooling2D()(x)
    x = L.Dense(128, activation='relu')(x)
    x = L.Dropout(0.4)(x)
    out = L.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out, name='Baseline_CNN')

model = build_cnn()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
      tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)]
t0 = time.time()
hist = model.fit(X_tr_aug, y_tr_aug, validation_data=(X_va, y_va), epochs=30, batch_size=128, callbacks=cb, verbose=2)
train_time = time.time() - t0
print(f'Training time: {train_time/60:.1f} min')

## 7. Results (tables and figures for the paper)

In [ ]:
# Training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(hist.history['accuracy'], label='train'); ax[0].plot(hist.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('Epoch'); ax[0].legend()
ax[1].plot(hist.history['loss'], label='train'); ax[1].plot(hist.history['val_loss'], label='val')
ax[1].set_title('Loss'); ax[1].set_xlabel('Epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('results/fig_cnn_training_curves.png', dpi=300); plt.show()

# Test metrics
y_pred = model.predict(X_te, batch_size=256).argmax(1)
acc = accuracy_score(y_te, y_pred)
f1m = f1_score(y_te, y_pred, average='macro')
print(f'Test accuracy: {acc:.4f} | Macro F1: {f1m:.4f}')
report = classification_report(y_te, y_pred, target_names=CLASSES, digits=4, output_dict=True)
pd.DataFrame(report).T.to_csv('results/table_cnn_per_class.csv')
print(classification_report(y_te, y_pred, target_names=CLASSES, digits=4))

# Confusion matrix (normalised)
cm = confusion_matrix(y_te, y_pred, normalize='true')
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Baseline CNN — normalised confusion matrix')
plt.tight_layout(); plt.savefig('results/fig_cnn_confusion_matrix.png', dpi=300); plt.show()

# Model size and inference time (needed for the "lightweight / fab-ready" claim)
model.save('results/baseline_cnn.keras')
size_mb = os.path.getsize('results/baseline_cnn.keras') / 1e6
_ = model.predict(X_te[:256], verbose=0)
t0 = time.time(); _ = model.predict(X_te[:1000], batch_size=1, verbose=0); ms = (time.time() - t0)
summary = {'model': 'Baseline CNN', 'params': int(model.count_params()), 'size_MB': round(size_mb, 2),
           'test_accuracy': round(acc, 4), 'macro_F1': round(f1m, 4),
           'inference_ms_per_wafer': round(ms, 3), 'train_time_min': round(train_time / 60, 1)}
pd.DataFrame([summary]).to_csv('results/table_model_comparison.csv', index=False)
summary

## 8. Save data for Part 2 and download results
Part 2 (ResNet-18, Vision Transformer, the hybrid CNN-Transformer, and Grad-CAM) reuses these exact splits, so all the models are compared fairly.

In [ ]:
np.savez_compressed('results/splits_64.npz', X_tr=X_tr_aug.argmax(-1).astype(np.uint8), y_tr=y_tr_aug,
                    X_va=X_va.argmax(-1).astype(np.uint8), y_va=y_va, X_te=X_te.argmax(-1).astype(np.uint8), y_te=y_te)
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')